# Full-run test for WGAN-GP

This notebook provides runnable cells to perform a full training run for 'WGAN-GP' using the project `TrainTestSplitPipeline`.

**WGAN-GP (Wasserstein GAN with Gradient Penalty)** is an improved GAN architecture that:
- Uses Wasserstein distance as the loss metric (more stable training)
- Enforces Lipschitz constraint via gradient penalty instead of weight clipping
- Provides better convergence and mode coverage

Notes:
- This implementation uses PyTorch
- Training can take significant time depending on dataset size and epochs
- GPU acceleration is automatically used if available

In [1]:
# Install PyTorch if needed
# Uncomment the appropriate line based on your system:

# For CPU only:
# !pip install torch torchvision torchaudio

# For CUDA 11.8:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# For CUDA 12.1:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [2]:
# Imports and helpers
import os
import importlib
import traceback
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

def ensure(path):
    os.makedirs(path, exist_ok=True)

def make_pipeline(model_callable, skip_evaluations=True, evaluations=None):
    if skip_evaluations:
        return TrainTestSplitPipeline(model=model_callable, evaluations=[], override_evaluations=True)
    elif evaluations is not None:
        return TrainTestSplitPipeline(model=model_callable, evaluations=evaluations, override_evaluations=True)
    else:
        return TrainTestSplitPipeline(model=model_callable)

In [3]:
# User configuration
DATASETS = ['adult', 'car', 'magic', 'nursery', 'shuttle']
SKIP_EVALUATIONS = False  # Set to True to skip TSTR evaluations

# Create directories
ensure('discretized_data')
ensure('sample_data')
ensure('synthetic')
ensure('Results')

MODEL_MAP = {
    'wgangp': ('katabatic.models.wgangp.models', 'WGANGPModel'),
}

# WGAN-GP Configuration
WGANGP_CONFIG = {
    'latent_dim': 128,
    'generator_dims': [256, 256],
    'critic_dims': [256, 256],
    'batch_size': 64,
    'epochs': 300,
    'critic_iterations': 5,
    'lambda_gp': 10.0,
    'learning_rate': 0.0001,
    'random_state': 42,
}

# Quick test configuration
WGANGP_CONFIG_QUICK = {
    'latent_dim': 64,
    'generator_dims': [128],
    'critic_dims': [128],
    'batch_size': 32,
    'epochs': 10,
    'critic_iterations': 3,
    'lambda_gp': 10.0,
    'learning_rate': 0.0001,
    'random_state': 42,
}

## Preprocess datasets (run once)
Run this cell to discretize the raw CSVs into `discretized_data/{dataset}.csv`.

In [4]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'Preprocessing {dataset}...')
    try:
        discretize_preprocess(
            file_path=f'raw_data/{dataset}.csv',
            output_path=f'discretized_data/{dataset}.csv',
            bins=10,
            strategy='uniform'
        )
        print(f'Discretized -> discretized_data/{dataset}.csv')
    except Exception as e:
        print(f'Failed to preprocess {dataset}: {e}')
        traceback.print_exc()


Preprocessing adult...
Preprocessing: raw_data/adult.csv
Saved preprocessed discrete dataset to: discretized_data/adult.csv
Discretized -> discretized_data/adult.csv

Preprocessing car...
Preprocessing: raw_data/car.csv
Saved preprocessed discrete dataset to: discretized_data/car.csv
Discretized -> discretized_data/car.csv

Preprocessing magic...
Preprocessing: raw_data/magic.csv
Saved preprocessed discrete dataset to: discretized_data/magic.csv
Discretized -> discretized_data/magic.csv

Preprocessing nursery...
Preprocessing: raw_data/nursery.csv
Saved preprocessed discrete dataset to: discretized_data/nursery.csv
Discretized -> discretized_data/nursery.csv

Preprocessing shuttle...
Preprocessing: raw_data/shuttle.csv
Saved preprocessed discrete dataset to: discretized_data/shuttle.csv
Discretized -> discretized_data/shuttle.csv


## Run full WGAN-GP
**Tip:** Start with `WGANGP_CONFIG_QUICK` for testing.

In [ ]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'WGAN-GP -> {dataset}')
    synth_dir = os.path.join('synthetic', dataset, 'wgangp')
    ensure(synth_dir)

    try:
        mod_path, cls_name = MODEL_MAP['wgangp']
        module = importlib.import_module(mod_path)
        ModelClass = getattr(module, cls_name)

        model_factory = lambda: ModelClass(**WGANGP_CONFIG)
        pipeline = make_pipeline(model_factory, skip_evaluations=SKIP_EVALUATIONS)

        pipeline.run(
            input_csv=f'discretized_data/{dataset}.csv',
            output_dir=f'sample_data/{dataset}',
            synthetic_dir=synth_dir,
            real_test_dir=f'sample_data/{dataset}'
        )

        print(f'WGAN-GP finished for {dataset}')
    except Exception as e:
        print(f'WGAN-GP failed for {dataset}: {e}')
        traceback.print_exc()


WGAN-GP -> adult
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
[WGAN-GP] Using device: cpu
[WGAN-GP] Starting training for 300 epochs...
[WGAN-GP] Dataset size: 26048, Batch size: 64
[WGAN-GP] Epoch [1/300] C_loss: -1.0706, G_loss: -0.0151, W_dist: 1.3573
[WGAN-GP] Epoch [50/300] C_loss: -0.2851, G_loss: 1.4916, W_dist: 0.3405
[WGAN-GP] Epoch [100/300] C_loss: -0.1870, G_loss: 0.9850, W_dist: 0.2320
[WGAN-GP] Epoch [150/300] C_loss: -0.1563, G_loss: 1.4107, W_dist: 0.1974
[WGAN-GP] Epoch [200/300] C_loss: -0.1167, G_loss: 1.2237, W_dist: 0.1631
[WGAN-GP] Epoch [250/300] C_loss: -0.1107, G_loss: 1.2324, W_dist: 0.1450
[WGAN-GP] Epoch [300/300] C_loss: -0.09

Traceback (most recent call last):
  File "C:\Users\lbrum\AppData\Local\Temp\ipykernel_15480\2032152655.py", line 15, in <module>
    pipeline.run(
  File "C:\Users\lbrum\OneDrive\Documents\Uni\2025\T3\SIT374\Katabatic\katabatic\pipeline\train_test_split\pipeline.py", line 46, in run
    eval_instance.evaluate()
  File "C:\Users\lbrum\OneDrive\Documents\Uni\2025\T3\SIT374\Katabatic\katabatic\evaluate\tstr\evaluation.py", line 59, in evaluate
    model.fit(x_train_scaled, self.y_train)
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1231, in fit
    check_classification_targets(y)
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\utils\multiclass.py", line 219, in check_classification_targets
    raise ValueError(
ValueError: Unknown label type: c


WGAN-GP -> car
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
[WGAN-GP] Using device: cpu
[WGAN-GP] Starting training for 300 epochs...
[WGAN-GP] Dataset size: 1382, Batch size: 64
[WGAN-GP] Epoch [1/300] C_loss: 2.1942, G_loss: -0.2349, W_dist: 0.0889
[WGAN-GP] Epoch [50/300] C_loss: -0.1305, G_loss: 0.5839, W_dist: 0.2189
[WGAN-GP] Epoch [100/300] C_loss: -0.3898, G_loss: 1.2303, W_dist: 0.5241
[WGAN-GP] Epoch [150/300] C_loss: -0.5393, G_loss: 2.6696, W_dist: 0.6597
[WGAN-GP] Epoch [200/300] C_loss: -0.5829, G_loss: 3.8570, W_dist: 0.6691
[WGAN-GP] Epoch [250/300] C_loss: -0.5564, G_loss: 3.9207, W_dist: 0.6389
[WGAN-G

Traceback (most recent call last):
  File "C:\Users\lbrum\AppData\Local\Temp\ipykernel_15480\2032152655.py", line 15, in <module>
    pipeline.run(
  File "C:\Users\lbrum\OneDrive\Documents\Uni\2025\T3\SIT374\Katabatic\katabatic\pipeline\train_test_split\pipeline.py", line 46, in run
    eval_instance.evaluate()
  File "C:\Users\lbrum\OneDrive\Documents\Uni\2025\T3\SIT374\Katabatic\katabatic\evaluate\tstr\evaluation.py", line 59, in evaluate
    model.fit(x_train_scaled, self.y_train)
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1231, in fit
    check_classification_targets(y)
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\utils\multiclass.py", line 219, in check_classification_targets
    raise ValueError(
ValueError: Unknown label type: c

Saved train/test full data
Train size: (15216, 11), Test size: (3804, 11)
Train label distribution:
 class
0    0.648396
1    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.648265
1    0.351735
Name: proportion, dtype: float64
Saved X/y split
Training shape: (15216, 10) (15216,)
Test shape: (3804, 10) (3804,)
[WGAN-GP] Using device: cpu
[WGAN-GP] Starting training for 300 epochs...
[WGAN-GP] Dataset size: 15216, Batch size: 64
[WGAN-GP] Epoch [1/300] C_loss: -0.6994, G_loss: -0.1906, W_dist: 1.0663
[WGAN-GP] Epoch [50/300] C_loss: -0.2125, G_loss: 0.6538, W_dist: 0.2540
[WGAN-GP] Epoch [100/300] C_loss: -0.1931, G_loss: 0.5956, W_dist: 0.2288
[WGAN-GP] Epoch [150/300] C_loss: -0.1760, G_loss: 1.3898, W_dist: 0.2000
[WGAN-GP] Epoch [200/300] C_loss: -0.1511, G_loss: 1.4696, W_dist: 0.1755
[WGAN-GP] Epoch [250/300] C_loss: -0.1313, G_loss: 1.3187, W_dist: 0.1550
[WGAN-GP] Epoch [300/300] C_loss: -0.1145, G_loss: 1.1662, W_dist: 0.1379
[WGAN-GP] Training 

Traceback (most recent call last):
  File "C:\Users\lbrum\AppData\Local\Temp\ipykernel_15480\2032152655.py", line 15, in <module>
    pipeline.run(
  File "C:\Users\lbrum\OneDrive\Documents\Uni\2025\T3\SIT374\Katabatic\katabatic\pipeline\train_test_split\pipeline.py", line 46, in run
    eval_instance.evaluate()
  File "C:\Users\lbrum\OneDrive\Documents\Uni\2025\T3\SIT374\Katabatic\katabatic\evaluate\tstr\evaluation.py", line 59, in evaluate
    model.fit(x_train_scaled, self.y_train)
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1231, in fit
    check_classification_targets(y)
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\utils\multiclass.py", line 219, in check_classification_targets
    raise ValueError(
ValueError: Unknown label type: c

Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
0    0.333333
1    0.329186
3    0.312018
4    0.025270
2    0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
0    0.333333
1    0.329090
3    0.312114
4    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)
[WGAN-GP] Using device: cpu
[WGAN-GP] Starting training for 300 epochs...
[WGAN-GP] Dataset size: 10368, Batch size: 64
[WGAN-GP] Epoch [1/300] C_loss: -1.2500, G_loss: -0.1731, W_dist: 1.7491
[WGAN-GP] Epoch [50/300] C_loss: -0.6039, G_loss: -0.3783, W_dist: 0.6708
[WGAN-GP] Epoch [100/300] C_loss: -0.4330, G_loss: -1.7163, W_dist: 0.4829
[WGAN-GP] Epoch [150/300] C_loss: -0.3975, G_loss: -2.1029, W_dist: 0.4398
[WGAN-GP] Epoch [200/300] C_loss: -0.3489, G_loss: -2.3655, W_dist: 0.3894
[WGAN-GP] Epoch [250/300] C_loss: -0.2889, G_loss: -0.1303, W_dist: 0.3271
[WGAN-GP] Epoch [300/300] C_lo

Traceback (most recent call last):
  File "C:\Users\lbrum\AppData\Local\Temp\ipykernel_15480\2032152655.py", line 15, in <module>
    pipeline.run(
  File "C:\Users\lbrum\OneDrive\Documents\Uni\2025\T3\SIT374\Katabatic\katabatic\pipeline\train_test_split\pipeline.py", line 46, in run
    eval_instance.evaluate()
  File "C:\Users\lbrum\OneDrive\Documents\Uni\2025\T3\SIT374\Katabatic\katabatic\evaluate\tstr\evaluation.py", line 59, in evaluate
    model.fit(x_train_scaled, self.y_train)
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1231, in fit
    check_classification_targets(y)
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\sklearn\utils\multiclass.py", line 219, in check_classification_targets
    raise ValueError(
ValueError: Unknown label type: c


WGAN-GP -> shuttle
Loaded data with shape: (58000, 10)
Saved train/test full data
Train size: (46400, 10), Test size: (11600, 10)
Train label distribution:
 class
0    0.785970
3    0.153491
4    0.056336
2    0.002953
1    0.000862
6    0.000216
5    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.785948
3    0.153534
4    0.056293
2    0.002931
1    0.000862
6    0.000259
5    0.000172
Name: proportion, dtype: float64
Saved X/y split
Training shape: (46400, 9) (46400,)
Test shape: (11600, 9) (11600,)
[WGAN-GP] Using device: cpu
[WGAN-GP] Starting training for 300 epochs...
[WGAN-GP] Dataset size: 46400, Batch size: 64
[WGAN-GP] Epoch [1/300] C_loss: -0.3400, G_loss: -0.2943, W_dist: 0.5144
[WGAN-GP] Epoch [50/300] C_loss: -0.1285, G_loss: -0.5043, W_dist: 0.1506
